# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

In [ ]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [ ]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [5]:
!pip install requests


   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   -------- ------------------------------- 1/5 [idna]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   -------------------------------- ------- 4/5 [requests]
   ---------------------------------------- 5/5 [requests]



In [6]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

def get_cat_fact(_):

    response = requests.get(CAT_API_URL)
    return response.json().get('fact')

# 1. POBIERANIE SEKWENCYJNE
print("Rozpoczynam pobieranie sekwencyjne...")
start_seq = time.time()
facts_seq = []
for i in range(20):
    facts_seq.append(get_cat_fact(i))
end_seq = time.time() - start_seq
print(f"Pobrano {len(facts_seq)} faktów sekwencyjnie w czasie: {end_seq:.4f} s")

# 2. POBIERANIE WIELOWĄTKOWE
print("\nRozpoczynam pobieranie wielowątkowe...")
start_threads = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    # Wersja ThreadPoolExecutor.map, która uruchomi get_cat_fact 20 razy i przechwyci listę wyników
    facts_threads = list(executor.map(get_cat_fact, range(20)))
end_threads = time.time() - start_threads
print(f"Pobrano {len(facts_threads)} faktów wielowątkowo w czasie: {end_threads:.4f} s")

# 3. PORÓWNANIE
print(f"\nRóżnica czasu wykonania: {end_seq - end_threads:.4f} s")
if end_threads < end_seq:
    print(f"Wersja wielowątkowa była {(end_seq/end_threads):.2f}x szybsza!")


Rozpoczynam pobieranie sekwencyjne...
Pobrano 20 faktów sekwencyjnie w czasie: 4.6046 s

Rozpoczynam pobieranie wielowątkowe...
Pobrano 20 faktów wielowątkowo w czasie: 0.6833 s

Różnica czasu wykonania: 3.9213 s
Wersja wielowątkowa była 6.74x szybsza!


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [1]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time


def producer(q, limit):
    print("[Producent] Zaczynam generować liczby...")
    for i in range(1, limit + 1):
        print(f"[Producent] Wrzucam do kolejki: {i}")
        q.put(i)
        # Niewielkie opóźnienie by urealnić "generowanie" danych
        time.sleep(0.2) 
        
    print("[Producent] Koniec generowania, wysyłam sygnał stopu.")
    q.put(None) # Sygnał zakończenia (poison pill)

def even_consumer(q):
    print("[Konsument 1] Gotowy na liczby PARZYSTE.")
    while True:
        item = q.get()
        if item is None:
            # Natrafiono na sygnał zakończenia - odłóżmy go by Konsument 2 też mógł wyjść
            q.put(None)
            break
            
        if item % 2 == 0:
            print(f"---> [Konsument 1: Parzyste] Przetwarzam: {item}")
        else:
            # To liczba nieparzysta - zwracamy ją z powrotem na koniec kolejki
            q.put(item)
            # Dajemy środowisku czas na przełączenie wątków (żeby Konsument 2 zdążył ją przejąć)
            time.sleep(0.05)

def odd_consumer(q):
    print("[Konsument 2] Gotowy na liczby NIEPARZYSTE.")
    while True:
        item = q.get()
        if item is None:
            q.put(None)
            break
            
        if item % 2 != 0:
            print(f"---> [Konsument 2: Nieparzyste] Przetwarzam: {item}")
        else:
            q.put(item)
            time.sleep(0.05)

# Tworzymy JEDNĄ wspólną kolejkę
shared_queue = queue.Queue()

# Inicjalizujemy wszystkie 3 wątki
prod_thread = threading.Thread(target=producer, args=(shared_queue, 10))
cons1_thread = threading.Thread(target=even_consumer, args=(shared_queue,))
cons2_thread = threading.Thread(target=odd_consumer, args=(shared_queue,))

# Uruchamiamy wątki (konsumenci mogą wystartować tacy sami jak producent i od razu zaczną czekać)
cons1_thread.start()
cons2_thread.start()
prod_thread.start()

# Blokujemy program "główny" by poczekał na zakończenie działania wątków konsumentów/producenta
prod_thread.join()
cons1_thread.join()
cons2_thread.join()

print("Wszystko przetworzone, wątki bezpiecznie zakończone!")


[Konsument 1] Gotowy na liczby PARZYSTE.
[Konsument 2] Gotowy na liczby NIEPARZYSTE.
[Producent] Zaczynam generować liczby...
[Producent] Wrzucam do kolejki: 1
---> [Konsument 2: Nieparzyste] Przetwarzam: 1
[Producent] Wrzucam do kolejki: 2
---> [Konsument 1: Parzyste] Przetwarzam: 2
[Producent] Wrzucam do kolejki: 3
---> [Konsument 2: Nieparzyste] Przetwarzam: 3
[Producent] Wrzucam do kolejki: 4
---> [Konsument 1: Parzyste] Przetwarzam: 4
[Producent] Wrzucam do kolejki: 5
---> [Konsument 2: Nieparzyste] Przetwarzam: 5
[Producent] Wrzucam do kolejki: 6
---> [Konsument 1: Parzyste] Przetwarzam: 6
[Producent] Wrzucam do kolejki: 7
---> [Konsument 2: Nieparzyste] Przetwarzam: 7
[Producent] Wrzucam do kolejki: 8
---> [Konsument 1: Parzyste] Przetwarzam: 8
[Producent] Wrzucam do kolejki: 9
---> [Konsument 2: Nieparzyste] Przetwarzam: 9
[Producent] Wrzucam do kolejki: 10
---> [Konsument 1: Parzyste] Przetwarzam: 10
[Producent] Koniec generowania, wysyłam sygnał stopu.
Wszystko przetworzone, 

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [2]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    # Określamy nasz duży zbiór liczb
    zakres = range(1, 10001)
    
    # (OPCJONALNIE) Mierzymy czas sekwencyjny by mieć porównanie:
    print("Rozpoczynam obliczenia sekwencyjne (jeden po drugim)...")
    start_seq = time.time()
    wyniki_seq = [calculate_power_sum(n) for n in zakres]
    czas_seq = time.time() - start_seq
    print(f"Czas sekwencyjny: {czas_seq:.4f} s")
    
    # OBLICZENIA WIELOPROCESOWE
    cores = multiprocessing.cpu_count()
    print(f"\nRozpoczynam obliczenia wieloprocesowe używając {cores} rdzeni procesora...")
    start_multi = time.time()
    
    # Otwieramy pulę procesów
    with multiprocessing.Pool(processes=cores) as pool:
        # Funkcja map automatycznie dzieli elementy z 'zakres' 
        # i dystrybuuje je do dostępnych rdzeni.
        wyniki_multi = pool.map(calculate_power_sum, zakres)
        
    czas_multi = time.time() - start_multi
    print(f"Czas wieloprocesowy: {czas_multi:.4f} s")
    print(f"\nZysk wydajnościowy: aplikacja wieloprocesowa wykonała się {czas_seq / czas_multi:.2f} razy szybciej!")


Rozpoczynam obliczenia sekwencyjne (jeden po drugim)...
Czas sekwencyjny: 0.6197 s

Rozpoczynam obliczenia wieloprocesowe używając 8 rdzeni procesora...
Czas wieloprocesowy: 0.5977 s

Zysk wydajnościowy: aplikacja wieloprocesowa wykonała się 1.04 razy szybciej!
